# Les Dataclasses en Python

Cette partie du cours couvre les bases des dataclasses en Python, introduites dans la version 3.7 via le module `dataclasses`. Les dataclasses sont des classes spéciales conçues principalement pour stocker des données, avec des méthodes automatisées comme `__init__`, `__repr__`, `__eq__`, etc. Elles simplifient la création de classes "data-oriented" en réduisant le boilerplate. Nous verrons la syntaxe, des exemples d'utilisation, et quelques exercices pour pratiquer.

En Python, importez le module avec `from dataclasses import dataclass`. Les dataclasses sont décorées avec `@dataclass` et utilisent `field` pour des options avancées. Elles héritent implicitement des fonctionnalités, rendant le code plus concis et lisible.



## 1. Définition Basique d'une Dataclass
Une dataclass simple se définit avec le décorateur `@dataclass` et des attributs de classe annotés avec des types (optionnel mais recommandé).

### Exemples
```python
from dataclasses import dataclass

# Définition d'une dataclass
@dataclass
class Personne:
    nom: str
    age: int
    ville: str = "Paris"  # Valeur par défaut

# Création d'une instance
p1 = Personne("Alice", 30)
p2 = Personne("Bob", 25, "Lyon")

# Accès aux attributs
print(p1.nom)  # Affiche "Alice"
print(p2.ville)  # Affiche "Lyon"

# Méthodes auto-générées
print(p1)  # Affiche Personne(nom='Alice', age=30, ville='Paris') via __repr__
print(p1 == p2)  # False via __eq__
```

Note : Sans `@dataclass`, vous devriez implémenter manuellement `__init__`, `__repr__`, etc. Les annotations de type aident à la lisibilité et aux outils comme mypy, mais ne sont pas obligatoires.



## 2. Options du Décorateur `@dataclass`
Le décorateur accepte des paramètres comme `init=True` (défaut), `repr=True`, `eq=True`, `order=False` (pour `__lt__` etc.), `frozen=False` (immutable), `slots=False` (pour `__slots__` en Python 3.10+).

### Exemples
```python
from dataclasses import dataclass

# Dataclass avec comparaison et ordre
@dataclass(order=True)
class Point:
    x: int
    y: int

p1 = Point(1, 2)
p2 = Point(3, 0)
print(p1 < p2)  # True (comparaison lexicographique)

# Dataclass frozen (immutable)
@dataclass(frozen=True)
class Config:
    chemin: str
    port: int = 8080

c = Config("/app")
# c.port = 9000  # Erreur : FrozenInstanceError
```

Note : `frozen=True` rend les instances immuables après création, idéal pour des hashes ou des clés de dict. `order=True` génère des méthodes de comparaison basées sur les champs.



## 3. Champs Avancés avec `field`
Utilisez `from dataclasses import field` pour personnaliser les champs : defaults factories, exclusion de `__init__`, comparaisons, etc.

### Exemples
```python
from dataclasses import dataclass, field
import datetime

@dataclass
class Produit:
    nom: str
    prix: float
    date_creation: datetime.datetime = field(default_factory=datetime.datetime.now)
    _interne: str = field(default="secret", init=False, repr=False)  # Non dans init/repr

prod = Produit("Ordinateur", 999.99)
print(prod)  # Affiche Produit(nom='Ordinateur', prix=999.99, date_creation=...) sans _interne
print(prod._interne)  # Affiche "secret"
```

Note : `default_factory` appelle une fonction pour la valeur par défaut (utile pour des listes ou dates). `init=False` exclut du constructeur ; `repr=False` du `__repr__` ; `compare=False` de `__eq__` et ordres.



## 4. Méthode `__post_init__`
Ajoutez une méthode `__post_init__` pour du traitement après l'initialisation automatique.

### Exemples
```python
from dataclasses import dataclass

@dataclass
class Rectangle:
    longueur: float
    largeur: float
    aire: float = 0.0  # Initialisé à 0

    def __post_init__(self):
        self.aire = self.longueur * self.largeur

rect = Rectangle(5, 3)
print(rect.aire)  # Affiche 15.0
```

Note : `__post_init__` est appelée après `__init__` généré. Utilisez-la pour des calculs dérivés ou validations.



## 5. Conversion et Utilitaires
Utilisez `asdict()`, `astuple()`, `replace()` pour convertir ou modifier des instances. Les dataclasses intègrent bien avec JSON via `json` module.

### Exemples
```python
from dataclasses import dataclass, asdict, astuple, replace
import json

@dataclass
class Utilisateur:
    id: int
    nom: str

u = Utilisateur(1, "Charlie")
print(asdict(u))  # {'id': 1, 'nom': 'Charlie'}
print(astuple(u))  # (1, 'Charlie')

u_modifie = replace(u, nom="David")
print(u_modifie)  # Utilisateur(id=1, nom='David')

json_str = json.dumps(asdict(u))  # '{"id": 1, "nom": "Charlie"}'
```

Note : Ces fonctions sont importées du module `dataclasses`. Pour JSON, gérez les types non sérialisables (ex. : datetime) avec des encodeurs custom.



## Vérification des Types
Les dataclasses sont des classes normales :
```python
from dataclasses import is_dataclass

print(is_dataclass(Personne))  # True
print(type(p1))  # <class '__main__.Personne'>
```



## Exercices
Voici quelques exercices simples pour pratiquer. Essayez de les résoudre dans un interpréteur Python, puis vérifiez les résultats.

1. **Exercice 1 : Dataclass basique**
   Définissez une dataclass `Livre` avec `titre: str`, `auteur: str`, `annee: int = 2023`. Créez une instance sans année et affichez-la.


In [1]:
from dataclasses import dataclass

@dataclass
class Livre:
    titre: str
    auteur: str
    annee: int = 2023

livre = Livre(titre="Python Avancé", auteur="Jacques", annee=2025)
print(livre)

Livre(titre='Python Avancé', auteur='Jacques', annee=2025)


2. **Exercice 2 : Frozen et order**
   Créez une dataclass `Coordonnee` avec `x: float`, `y: float`, frozen et order=True. Créez deux instances et comparez-les avec `<`.



In [6]:
@dataclass(frozen=True, order=True)
class Coordonnee:
    x: float
    y: float

point_a = Coordonnee(1, 1000)
point_b = Coordonnee(2, 1)

# point_a.x = 3
print(point_a < point_b)

True


In [7]:
 point_a.x = 3

FrozenInstanceError: cannot assign to field 'x'

3. **Exercice 3 : Field avancé**
   Dans une dataclass `Tache`, ajoutez `description: str`, `priorite: int`, et un champ `echeance: list = field(default_factory=list)`. Créez une instance et ajoutez un élément à la liste.



In [12]:
from dataclasses import field

@dataclass
class Tache:
    titre: str
    description: str
    priorite: int
    echeance: list[str] = field(default_factory=list)

tache_1 = Tache(
    titre="Ajouter un modèle de données clients",
    description="""
    Créer une dataclass pour représenter un client en se basant sur le fichier base.json
    """,
    priorite=1,
    )

tache_1.echeance.append("2025-10-01")
tache_1.echeance.append("2025-11-01")

print(tache_1)

Tache(titre='Ajouter un modèle de données clients', description='\n    Créer une dataclass pour représenter un client en se basant sur le fichier base.json\n    ', priorite=1, echeance=['2025-10-01', '2025-11-01'])


j4. **Exercice 4 : __post_init__**
   Définissez `Cercle` avec `rayon: float`, `circonference: float = 0`. Dans `__post_init__`, calculez la circonférence (2 * pi * rayon). Créez et affichez.



In [14]:
import math
from dataclasses import field

@dataclass
class Cercle:
    rayon: float
    circonference: float = field(default=0, init=False)

    def __post_init__(self):
        self.circonference = 2 * math.pi * self.rayon


cercle = Cercle(10)

print(cercle)

62.83185307179586


5. **Exercice 5 : Conversion**
   Utilisez `asdict` sur une instance de `Personne` (ex. 1), modifiez le dict, puis créez une nouvelle instance avec `replace`. Affichez en JSON.


In [17]:
from dataclasses import asdict
import json
from pathlib import Path

@dataclass
class Personne:
    nom: str
    age: int
    ville: str = "Paris"

# Création d'une instance
p1 = Personne("Alice", 30)
p2 = Personne("Bob", 25, "Lyon")

p1_dict = asdict(p1)
p2_dict = asdict(p2)

print(p1_dict)
print(p2_dict)

p1_txt = json.dumps(p1_dict)
path = Path("personne.json")

path.write_text(p1_txt)

{'nom': 'Alice', 'age': 30, 'ville': 'Paris'}
{'nom': 'Bob', 'age': 25, 'ville': 'Lyon'}


45

In [ ]:
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Produit:
    id: int
    nom: str
    prix: float
    quantite: int

    def prix_total(self) -> float:
        return self.prix * self.quantite


@dataclass
class Commande:
    id: int
    produits: list[Produit]
    date: datetime = datetime.now()

    def pix_total(self):
        return sum([produit.prix_total() for produit in self.produits])

@dataclass
class Client:
    id: int
    nom: str
    adresse: str
    telephone: str
    email: str


@dataclass
class NotifierClient:
    client: Client
    commande: Commande

    def send_notification(self):
        return f"Commande num: {self.commande.id}, montant total: {self.commande.pix_total()} envoyée à l'adresse {self.client.email}"


In [ ]:
from dataclasses import field

@dataclass
class Employe:
    id: int
    nom: str
    prenom: str
    salaire: float
    direction: str
    date_embauche: datetime



@dataclass
class Manager(Employe):
    classification: str = field(default="G", init=False)
    poste: str


@dataclass
class Stagiaire(Employe):
    classification: str = field(default="S", init=False)
    poste: str